# 08 — Download Mel+Nevus Histo Dataset

**Issue:** D4.1 (#137)  
**Dataset:** Melanoma and Nevus Dermoscopy Images with Confirmed Histopathological Diagnosis  
**ISIC Collection ID:** 294  
**Expected:** 18,133 images (7,191 melanoma, 39.7%)  
**Destination:** `data/mel_nevus_histo/`

In [ ]:
import pandas as pd
from pathlib import Path

METADATA_PATH = Path("../../../../data/mel_nevus_histo/metadata.csv")
IMAGES_DIR = Path("../../../../data/mel_nevus_histo/images")

assert METADATA_PATH.exists(), f"Metadata not found at {METADATA_PATH}"
assert IMAGES_DIR.exists(), f"Images dir not found at {IMAGES_DIR}"

## 1. Metadata — row count assert

In [ ]:
df = pd.read_csv(METADATA_PATH)
print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")

assert len(df) == 18_133, f"Expected 18,133 rows, got {len(df)}"
print("\nRow count OK — 18,133")

## 2. Class distribution — diagnosis_2 / diagnosis_3

In [ ]:
print("=== diagnosis_2 ===")
print(df["diagnosis_2"].value_counts(dropna=False).to_string())
print()
print("=== diagnosis_3 ===")
print(df["diagnosis_3"].value_counts(dropna=False).to_string())

In [ ]:
# Melanoma count from diagnosis_3
mel_mask = df["diagnosis_3"].str.lower().str.contains("melanoma", na=False)
mel_count = mel_mask.sum()
mel_pct = mel_count / len(df) * 100
print(f"Melanoma rows: {mel_count:,} ({mel_pct:.1f}%)")
print(f"Expected:      7,191 (39.7%)")

## 3. Image count and zero-byte check

In [ ]:
images = list(IMAGES_DIR.glob("*.jpg")) + list(IMAGES_DIR.glob("*.JPG"))
print(f"Images on disk: {len(images):,}")
assert len(images) == 18_133, f"Expected 18,133 images, got {len(images)}"
print("Image count OK — 18,133")

In [ ]:
zero_byte = [p for p in images if p.stat().st_size == 0]
print(f"Zero-byte files: {len(zero_byte)}")
assert len(zero_byte) == 0, f"Found {len(zero_byte)} zero-byte files: {zero_byte[:5]}"
print("Zero-byte check OK")

## 4. Summary table

In [ ]:
bcn_overlap_estimate = 5_633
post_filter_estimate = len(df) - bcn_overlap_estimate
post_filter_mel_estimate = mel_count - (mel_count - 4_334)  # ~4,334 unique mel after dedup

summary = pd.DataFrame({
    "Stage": ["Raw MNH", "After BCN20000 dedup (est.)"],
    "Total images": [f"{len(df):,}", f"~{post_filter_estimate:,}"],
    "Melanoma": [f"{mel_count:,}", "~4,334"],
    "Melanoma %": [f"{mel_pct:.1f}%", "~34.7%"],
})
print(summary.to_string(index=False))